# PDF Exam Practice Bot
Trains an LSTM + GRU model on PDF text, generates questions from the content,
takes your answer, and checks correctness using cosine similarity on model-encoded vectors.

In [ ]:
!pip install sentence-transformers tensorflow pypdf numpy -q

In [23]:
!pip install pypdf -q
import re
import random
import numpy as np

from pypdf import PdfReader

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Input , Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics.pairwise import cosine_similarity

In [15]:
# ── Configuration ─────────────────────────────────────────
PDF_PATH      = "/content/budget_speech.pdf"   # change to your PDF
NUM_QUESTIONS = 10
MAX_WORDS     = 5000   # tokenizer vocabulary size
MAX_LEN       = 30     # max tokens per sentence
EMBED_DIM     = 64
LSTM_UNITS    = 64
GRU_UNITS     = 32
EPOCHS        = 20
BATCH_SIZE    = 8


In [16]:
reader = PdfReader(PDF_PATH)

text = ""
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text

print(f"PDF Loaded Successfully ✅  ({len(text):,} characters)")


PDF Loaded Successfully ✅  (102,141 characters)


In [17]:
# Split into sentences and keep only meaningful ones (>6 words)
raw_sentences = re.split(r'(?<=[.?!])\s+', text)

sentences = []
for s in raw_sentences:
    s = s.strip()
    s = re.sub(r'\s+', ' ', s)          # collapse whitespace
    if len(s.split()) > 6:              # skip very short fragments
        sentences.append(s)

print(f"Total usable sentences: {len(sentences)}")
print("Sample:", sentences[0][:120])


Total usable sentences: 597
Sample: GOVERNMENT OF INDIA BUDGET 2026-2027 SPEECH OF NIRMALA SITHARAMAN MINISTER OF FINANCE February 1, 2026 CONTENTS Page No.


In [21]:
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

# Labels: 1 for every real sentence (self-supervised; model learns sentence representations)
y = np.ones(len(X))

print(f"X shape: {X.shape}")


X shape: (597, 30)


In [24]:
base_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM),
    Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)),
    Bidirectional(GRU(GRU_UNITS, return_sequences=True)),
    GRU(GRU_UNITS),
    Dense(1, activation='sigmoid')
])

base_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

base_model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 30, 64)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 30, 64)         │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 426,593 (1.63 MB)

 Trainable params: 426,593 (1.63 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
base_model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\nLSTM + GRU Model Trained ✅")


Epoch 1/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 18s 62ms/step - accuracy: 0.9983 - loss: 0.0720
Epoch 2/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 80ms/step - accuracy: 1.0000 - loss: 4.7654e-04
Epoch 3/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 61ms/step - accuracy: 1.0000 - loss: 3.5880e-04
Epoch 4/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 78ms/step - accuracy: 1.0000 - loss: 2.7876e-04
Epoch 5/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 1.0000 - loss: 2.2251e-04
Epoch 6/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 1.0000 - loss: 1.8173e-04
Epoch 7/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 98ms/step - accuracy: 1.0000 - loss: 1.5111e-04
Epoch 8/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 1.0000 - loss: 1.2750e-04
Epoch 9/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 12s 96ms/step - accuracy: 1.0000 - loss: 1.0897e-04
Epoch 10/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 11s 111ms/step - accuracy: 1.0000 - loss: 9.4198e-05
Epoch 11/20
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 62ms/step - accuracy: 1.0000 - loss: 8.2241e-05
Epoch 12/20
75/75 

In [26]:
# Build encoder: same weights, output is the GRU pooled vector (before Dense)
encoder = Model(
    inputs=base_model.inputs,
    outputs=base_model.layers[-2].output   # GRU output
)

def encode_text(text_input):
    """Tokenise → pad → encode with LSTM+GRU → return vector."""
    seq = tokenizer.texts_to_sequences([text_input])
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    return encoder.predict(seq, verbose=0)   # shape (1, GRU_UNITS)

print("Encoder ready ✅")

Encoder ready ✅


In [27]:
# Select NUM_QUESTIONS sentences as questions.
# Each sentence IS both the question and the model (reference) answer.
exam_questions = random.sample(sentences, min(NUM_QUESTIONS, len(sentences)))

# Pre-encode the model answers once
model_answer_vectors = [encode_text(q) for q in exam_questions]

print(f"{len(exam_questions)} exam questions ready ✅")


10 exam questions ready ✅


In [28]:
print("\n========== EXAM START ==========\n")

score = 0

for i, question in enumerate(exam_questions, 1):

    print(f"\nQuestion {i}/{len(exam_questions)}:")
    print(question)

    user_answer = input("\nYour Answer: ").strip()

    if not user_answer:
        print("(skipped)")
        continue

    # Encode user answer with the LSTM+GRU model
    user_vec   = encode_text(user_answer)
    model_vec  = model_answer_vectors[i - 1]

    # Cosine similarity between the two encoded vectors
    similarity = cosine_similarity(user_vec, model_vec)[0][0]

    print(f"\nCosine Similarity: {similarity:.4f}  (threshold: 0.65)")

    if similarity >= 0.65:
        print("👍 Correct")
        score += 1
    else:
        print("❌ Wrong")
        print("\nCorrect Answer:")
        print(question)

print("\n")



========== EXAM START ==========


Question 1/10:
Once applied by an IT Services company, the same safe harbour can be continued for a period of 5 years at a stretch at its choice.

Your Answer: me

Cosine Similarity: 1.0000  (threshold: 0.65)
👍 Correct

Question 2/10:
Change in taxation of buyback was brought in to address the improper use of buyback route by promoters.

Your Answer: o

Cosine Similarity: 1.0000  (threshold: 0.65)
👍 Correct

Question 3/10:
COOPERATIVES (i) Deduction of profit and gains to a primary co -operative where they supply cattle feed and cotton seed to a federal co-operative.

Your Answer: paul

Cosine Similarity: 1.0000  (threshold: 0.65)
👍 Correct

Question 4/10:
I propose to integrate assessment & penalty proceedings by way of a common order for both.

Your Answer: j

Cosine Similarity: 1.0000  (threshold: 0.65)
👍 Correct

Question 5/10:
During this past decade our Government has undertaken several initiatives for large-scale enhancement of public infrastr

KeyboardInterrupt: Interrupted by user

In [ ]:
total = len(exam_questions)
pct   = score / total * 100
bar   = '█' * score + '░' * (total - score)

print("========== RESULTS ==========")
print(f"Final Score : {score}/{total}")
print(f"Percentage  : {pct:.0f}%")
print(f"[{bar}]")
